# Task B -- one-layer versus no-layer reinitialization

Run 5 found that resetting only MuRIL's final encoder layer scored 0.6102 OOF macro-F1. This ablation tests whether preserving the final layer as well is better: `--reinit-layers 0` keeps both layers 11 and 12 from the TAPT checkpoint.

The control is `--reinit-layers 1`; the variant is `--reinit-layers 0`. Both use one shared TAPT checkpoint, the same deduplicated Task B data, fixed five-fold split (split seed 42), model seeds 43 and 44, six epochs, mean+max pooling, FGM, EMA, class weighting, no auxiliary head, and `--select last`.

Expected runtime: about 6--7 hours on a T4, including one TAPT pass. Upload this notebook to Kaggle with a GPU and Internet enabled, then use **Save Version -> Save & Run All**. This is a local ablation, not a submission notebook.

In [ ]:
import os, pathlib, shutil, subprocess, sys

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build one shared TAPT checkpoint

TAPT is run once and reused by all four classifier runs. The default settings match the earlier reinitialization experiments.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-muril"
TAPT_LOG = "artifacts/logs/no_reinit_tapt.log"
if pathlib.Path(TAPT_OUT).is_dir():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert pathlib.Path(TAPT_OUT).is_dir(), "TAPT checkpoint was not written"
print("TAPT checkpoint ready:", TAPT_OUT)


## 2. Run the one-layer control and no-reinitialization variant

The four tags are separate so the notebook can report seed 43 and seed 44 independently.

In [ ]:
COMMON = [
    "--model", TAPT_OUT,
    "--folds", "5",
    "--epochs", "6",
    "--select", "last",
    "--aux-weight", "0",
]
ARMS = [
    ("b_tapt_reinit1_vs0_s43", "1", "43"),
    ("b_tapt_reinit0_s43", "0", "43"),
    ("b_tapt_reinit1_vs0_s44", "1", "44"),
    ("b_tapt_reinit0_s44", "0", "44"),
]
for tag, n_layers, seed in ARMS:
    run([sys.executable, "-u", "-m", "hastika.task_b.train",
         "--tag", tag, *COMMON, "--seeds", seed,
         "--reinit-layers", n_layers], log=f"artifacts/logs/{tag}.log")
print("all four no-reinitialization ablation runs completed")


## 3. Recompute and compare OOF scores

This cell independently recomputes macro-F1 from each saved OOF probability matrix. It reports each seed, the mean of the two seed scores, and the score after averaging the two seeds' probabilities, plus per-class F1.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from hastika.common.preprocessing import dedupe_index

LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
train = pd.read_csv("data/raw/multiclass_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Hate Category"].tolist(), "task B")
train = train.iloc[keep].reset_index(drop=True)
y = train["Hate Category"].map(LABELS.index).to_numpy()

records, probabilities = [], {}
for tag, n_layers, seed in ARMS:
    probs = np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
    assert probs.shape == (len(y), len(LABELS)), f"unexpected OOF shape for {tag}: {probs.shape}"
    n_layers, seed = int(n_layers), int(seed)
    probabilities[(n_layers, seed)] = probs
    pred = probs.argmax(1)
    report = classification_report(y, pred, labels=range(len(LABELS)),
                                  target_names=LABELS, output_dict=True, zero_division=0)
    records.append({"layers": n_layers, "seed": seed,
                    "macro_f1": f1_score(y, pred, average="macro"),
                    **{f"f1_{label}": report[label]["f1-score"] for label in LABELS}})

by_seed = pd.DataFrame(records).sort_values(["seed", "layers"])
display(by_seed.round(4))

aggregate = []
for n_layers in [1, 0]:
    seed_scores = by_seed.loc[by_seed["layers"] == n_layers, "macro_f1"]
    mean_probs = np.mean([probabilities[(n_layers, seed)] for seed in [43, 44]], axis=0)
    aggregate.append({"reinit_layers": n_layers,
                      "mean_seed_macro_f1": seed_scores.mean(),
                      "mean_probability_macro_f1": f1_score(y, mean_probs.argmax(1), average="macro")})
aggregate = pd.DataFrame(aggregate).set_index("reinit_layers").loc[[1, 0]]
display(aggregate.round(4))
print(f"\nZero-layer minus one-layer, mean seed scores: {aggregate.loc[0, 'mean_seed_macro_f1'] - aggregate.loc[1, 'mean_seed_macro_f1']:+.4f}")
print(f"Zero-layer minus one-layer, averaged probabilities: {aggregate.loc[0, 'mean_probability_macro_f1'] - aggregate.loc[1, 'mean_probability_macro_f1']:+.4f}")

print("\nPer-class reports from averaged probabilities:")
for n_layers in [1, 0]:
    mean_probs = np.mean([probabilities[(n_layers, seed)] for seed in [43, 44]], axis=0)
    print(f"\nreinit layers = {n_layers}")
    print(classification_report(y, mean_probs.argmax(1), labels=range(len(LABELS)),
                                target_names=LABELS, digits=3, zero_division=0))


## 4. Preserve the ablation outputs

Download the copied directory from Kaggle. If zero-layer reinitialization wins consistently, update the recipe and use `--reinit-layers 0` for the final full-data fit.

In [ ]:
OUT = pathlib.Path("/kaggle/working/no_reinit_ablation_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag, _, _ in ARMS:
    src = pathlib.Path("artifacts/runs") / tag
    dst = OUT / tag
    dst.mkdir(parents=True, exist_ok=True)
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        if (src / name).exists():
            shutil.copy2(src / name, dst / name)
    shutil.copy2(pathlib.Path("artifacts/logs") / f"{tag}.log", OUT / f"{tag}.log")
if pathlib.Path(TAPT_LOG).exists():
    shutil.copy2(TAPT_LOG, OUT / pathlib.Path(TAPT_LOG).name)
by_seed.to_csv(OUT / "no_reinit_by_seed.csv", index=False)
aggregate.to_csv(OUT / "no_reinit_aggregate.csv")
print("saved outputs to", OUT)
